# Project Nexus – Batch Pipeline Logging

Reusable logging framework for the Bronze → Silver → Gold batch pipeline, written in **Scala on Apache Spark**.

Log tables (Delta):

| Table | Grain | Purpose |
| --- | --- | --- |
| `pipeline_run_log` | one row per run event | Run started / succeeded / failed |
| `pipeline_step_log` | one row per step execution | Layer, source, target, status, duration, row metrics |
| `pipeline_error_log` | one row per error | Error class, message, stack trace |

Usage pattern: create one `PipelineLogger` per run, wrap each step with `runStep`, then call `succeed()` or `fail()`.

## 1. Configuration

In [ ]:
object LogConfig {
  // Change catalog/schema to match your environment (Unity Catalog: catalog.schema.table)
  val catalog = "nexus"
  val schema  = "logging"

  val runTable   = s"$catalog.$schema.pipeline_run_log"
  val stepTable  = s"$catalog.$schema.pipeline_step_log"
  val errorTable = s"$catalog.$schema.pipeline_error_log"
}

## 2. Imports and log record definitions

In [ ]:
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
import java.sql.Timestamp
import java.util.UUID
import scala.reflect.runtime.universe.TypeTag
import scala.util.{Failure, Success, Try}

case class RunLogRecord(
  run_id: String,
  pipeline_name: String,
  event: String,                  // STARTED | SUCCEEDED | FAILED
  event_time: Timestamp,
  duration_seconds: Option[Long],
  triggered_by: String,
  message: String
)

case class StepLogRecord(
  run_id: String,
  pipeline_name: String,
  step_name: String,
  layer: String,                  // BRONZE | SILVER | GOLD | VALIDATION | OTHER
  source: String,
  target: String,
  status: String,                 // SUCCESS | FAILED
  start_time: Timestamp,
  end_time: Timestamp,
  duration_seconds: Long,
  rows_read: Long,
  rows_written: Long,
  rows_rejected: Long,
  message: String
)

case class ErrorLogRecord(
  run_id: String,
  pipeline_name: String,
  step_name: String,
  error_class: String,
  error_message: String,
  stack_trace: String,
  logged_at: Timestamp
)

case class StepMetrics(
  rowsRead: Long = 0L,
  rowsWritten: Long = 0L,
  rowsRejected: Long = 0L,
  message: String = ""
)

## 3. Create log tables

In [ ]:
spark.sql(s"CREATE SCHEMA IF NOT EXISTS ${LogConfig.catalog}.${LogConfig.schema}")

spark.sql(s'''
  CREATE TABLE IF NOT EXISTS ${LogConfig.runTable} (
    run_id           STRING,
    pipeline_name    STRING,
    event            STRING,
    event_time       TIMESTAMP,
    duration_seconds BIGINT,
    triggered_by     STRING,
    message          STRING
  ) USING DELTA
''')

spark.sql(s'''
  CREATE TABLE IF NOT EXISTS ${LogConfig.stepTable} (
    run_id           STRING,
    pipeline_name    STRING,
    step_name        STRING,
    layer            STRING,
    source           STRING,
    target           STRING,
    status           STRING,
    start_time       TIMESTAMP,
    end_time         TIMESTAMP,
    duration_seconds BIGINT,
    rows_read        BIGINT,
    rows_written     BIGINT,
    rows_rejected    BIGINT,
    message          STRING
  ) USING DELTA
''')

spark.sql(s'''
  CREATE TABLE IF NOT EXISTS ${LogConfig.errorTable} (
    run_id        STRING,
    pipeline_name STRING,
    step_name     STRING,
    error_class   STRING,
    error_message STRING,
    stack_trace   STRING,
    logged_at     TIMESTAMP
  ) USING DELTA
''')

## 4. PipelineLogger

In [ ]:
class PipelineLogger(
    spark: SparkSession,
    val pipelineName: String,
    triggeredBy: String = "manual"
) {
  import spark.implicits._

  val runId: String = UUID.randomUUID().toString
  private val runStartMs: Long = System.currentTimeMillis()

  private def now: Timestamp = new Timestamp(System.currentTimeMillis())

  private def append[T <: Product : TypeTag](row: T, table: String): Unit =
    Seq(row).toDF().write.format("delta").mode("append").saveAsTable(table)

  private def seconds(fromMs: Long): Long = (System.currentTimeMillis() - fromMs) / 1000

  // ---- Run level -------------------------------------------------------
  def startRun(message: String = ""): Unit = {
    println(s"[RUN START] $pipelineName run_id=$runId")
    append(RunLogRecord(runId, pipelineName, "STARTED", now, None, triggeredBy, message), LogConfig.runTable)
  }

  def succeed(message: String = ""): Unit = {
    val d = seconds(runStartMs)
    println(s"[RUN SUCCESS] $pipelineName run_id=$runId duration=${d}s")
    append(RunLogRecord(runId, pipelineName, "SUCCEEDED", now, Some(d), triggeredBy, message), LogConfig.runTable)
  }

  def fail(message: String): Unit = {
    val d = seconds(runStartMs)
    println(s"[RUN FAILED] $pipelineName run_id=$runId duration=${d}s message=$message")
    append(RunLogRecord(runId, pipelineName, "FAILED", now, Some(d), triggeredBy, message), LogConfig.runTable)
  }

  // ---- Error level -----------------------------------------------------
  def logError(stepName: String, e: Throwable): Unit =
    append(
      ErrorLogRecord(
        runId, pipelineName, stepName,
        e.getClass.getName,
        Option(e.getMessage).getOrElse(""),
        e.getStackTrace.take(15).mkString("\n"),
        now
      ),
      LogConfig.errorTable
    )

  // ---- Step level ------------------------------------------------------
  // Wraps a step: times it, records metrics, logs failures, then rethrows so the run stops.
  def runStep(stepName: String, layer: String, source: String, target: String)
             (body: => StepMetrics): StepMetrics = {
    val startTs = now
    val startMs = System.currentTimeMillis()
    println(s"[STEP START] $layer | $stepName | $source -> $target")

    Try(body) match {
      case Success(m) =>
        val d = seconds(startMs)
        append(
          StepLogRecord(runId, pipelineName, stepName, layer, source, target, "SUCCESS",
            startTs, now, d, m.rowsRead, m.rowsWritten, m.rowsRejected, m.message),
          LogConfig.stepTable
        )
        println(s"[STEP OK] $stepName read=${m.rowsRead} written=${m.rowsWritten} rejected=${m.rowsRejected} (${d}s)")
        m

      case Failure(e) =>
        val d = seconds(startMs)
        append(
          StepLogRecord(runId, pipelineName, stepName, layer, source, target, "FAILED",
            startTs, now, d, 0L, 0L, 0L, Option(e.getMessage).getOrElse("")),
          LogConfig.stepTable
        )
        logError(stepName, e)
        println(s"[STEP FAILED] $stepName: ${e.getMessage}")
        throw e
    }
  }
}

## 5. Example – wrapping a batch run

The steps below use generated data so the notebook runs end to end. Replace the bodies with your real Bronze, Silver and Gold logic. Each body must return a `StepMetrics`.

In [ ]:
val logger = new PipelineLogger(spark, pipelineName = "nexus_batch_pipeline", triggeredBy = "manual")
logger.startRun("Demo batch run")

try {
  // ---- Bronze ----
  val bronzeDf = spark.range(0, 1000).toDF("order_id")
    .withColumn("amount", (rand() * 500).cast("double"))

  logger.runStep("ingest_orders", "BRONZE", "adls://raw/orders.csv", "bronze.orders") {
    val n = bronzeDf.count()
    // bronzeDf.write.format("delta").mode("append").saveAsTable("bronze.orders")
    StepMetrics(rowsRead = n, rowsWritten = n)
  }

  // ---- Silver ----
  logger.runStep("validate_orders", "SILVER", "bronze.orders", "silver.orders") {
    val valid    = bronzeDf.filter(col("amount") > 10)
    val rejected = bronzeDf.filter(col("amount") <= 10)
    val v = valid.count()
    val r = rejected.count()
    // valid.write.format("delta").mode("overwrite").saveAsTable("silver.orders")
    // rejected.write.format("delta").mode("append").saveAsTable("silver.orders_rejected")
    StepMetrics(rowsRead = v + r, rowsWritten = v, rowsRejected = r, message = "amount > 10 rule")
  }

  // ---- Gold ----
  logger.runStep("build_fact_orders", "GOLD", "silver.orders", "gold.fact_orders") {
    val gold = bronzeDf.filter(col("amount") > 10)
    val n = gold.count()
    StepMetrics(rowsRead = n, rowsWritten = n)
  }

  logger.succeed("All steps completed")
} catch {
  case e: Throwable =>
    logger.fail(Option(e.getMessage).getOrElse(e.getClass.getName))
    throw e
}

## 6. Real-pipeline template

In [ ]:
// Copy this pattern for each real step:
//
// logger.runStep("silver_customers", "SILVER", "bronze.customers", "silver.customers") {
//   val src       = spark.table("bronze.customers")
//   val cleaned   = src.dropDuplicates("customer_id").filter(col("email").isNotNull)
//   val rejected  = src.join(cleaned, Seq("customer_id"), "left_anti")
//   cleaned.write.format("delta").mode("overwrite").saveAsTable("silver.customers")
//   rejected.write.format("delta").mode("append").saveAsTable("silver.customers_rejected")
//   StepMetrics(src.count(), cleaned.count(), rejected.count())
// }

## 7. Query the logs

In [ ]:
println("Run log")
spark.table(LogConfig.runTable)
  .orderBy(col("event_time").desc)
  .show(20, truncate = false)

In [ ]:
println("Step log for latest run")
val latestRun = spark.table(LogConfig.runTable)
  .filter(col("event") === "STARTED")
  .orderBy(col("event_time").desc)
  .select("run_id").head().getString(0)

spark.table(LogConfig.stepTable)
  .filter(col("run_id") === latestRun)
  .select("layer", "step_name", "status", "duration_seconds", "rows_read", "rows_written", "rows_rejected")
  .orderBy("start_time")
  .show(false)

In [ ]:
println("Rejection rate by step")
spark.table(LogConfig.stepTable)
  .filter(col("rows_read") > 0)
  .withColumn("rejection_pct", round(col("rows_rejected") / col("rows_read") * 100, 2))
  .select("run_id", "step_name", "rows_read", "rows_rejected", "rejection_pct")
  .orderBy(col("rejection_pct").desc)
  .show(20, truncate = false)

In [ ]:
println("Errors")
spark.table(LogConfig.errorTable)
  .orderBy(col("logged_at").desc)
  .select("logged_at", "step_name", "error_class", "error_message")
  .show(20, truncate = false)

## Notes

- Log writes are append-only Delta, so concurrent steps and reruns never overwrite history.
- `runStep` rethrows after logging, so a failed step stops the run and the outer `catch` records the run as `FAILED`.
- Row counts use `count()`, which triggers a Spark action. Cache the DataFrame if it feeds several metrics.
- Outside Unity Catalog, set `LogConfig.catalog` to `spark_catalog` or drop the catalog prefix.